In [ ]:
import os
os.chdir('/home/cbn-gpu12/FNF/VLM/LLaVA/LLaVA-Med')


from peft import PeftModel
from huggingface_hub import create_repo

import sys
import warnings
warnings.filterwarnings("ignore")
import random
import torch
from torch.utils.data.dataset import Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from io import BytesIO
import requests
from datetime import datetime
import gc
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
import json
import time 
from collections import defaultdict 
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, LoraModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.conversation import Conversation
from llava.mm_utils import tokenizer_image_token, process_images
from llava.model.builder import load_pretrained_model
from llava.conversation import conv_templates
from tqdm import tqdm
from torch.utils.data import DataLoader
import glob
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from multiprocessing import Pool, cpu_count
import pydicom
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import uuid
import pickle
from transformers import LlamaTokenizer
from llava.model import LlavaMistralForCausalLM  # LLaVA-Med의 모델 클래스 import
from dotenv import load_dotenv
import wandb
from functools import lru_cache
from torchvision.transforms import Resize, Compose, ToTensor
from functools import partial  # collate_fn에 인자 전달을 위한 패키지
from torch.utils.data._utils.pin_memory import pin_memory
from transformers import AutoTokenizer, AutoModelForCausalLM
from llava.utils import disable_torch_init
from accelerate import init_empty_weights
from accelerate import Accelerator 
from transformers import BitsAndBytesConfig
import cv2  # OpenCV를 활용한 빠른 이미지 저장
from huggingface_hub import notebook_login
from accelerate.utils import set_module_tensor_to_device 
load_dotenv()
os.environ["WANDB_API_KEY"] = ""
os.environ["HUGGING_FACE_HUB_TOKEN"] = ""
notebook_login()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 1:
    print(f"🖥 Using {torch.cuda.device_count()} GPUs: [0, 1]")

    
CACHE_DIR = "/home/cbn-gpu12/FNF/VLM/LLavA/dataset/cache"
os.makedirs(CACHE_DIR, exist_ok=True)


🖥 Using 4 GPUs: [0, 1]


In [ ]:
class ModelConfig:
    model_path: str = "microsoft/llava-med-v1.5-mistral-7b"
    max_length: int = 4096
    load_in_8bit: bool = True
    cache_dir: str = CACHE_DIR
    use_safetensors: bool = True
    

In [ ]:
def setup_gpu_environment():
    """GPU 환경 설정 및 CUDA 설정"""
    # CUDA 환경 변수 설정
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    
    # GPU 메모리 정리
    torch.cuda.empty_cache()
    gc.collect()
    
    # GPU 정보 출력
    gpu_count = torch.cuda.device_count()
    print(f"사용 가능한 GPU: {gpu_count}개")
    for i in range(gpu_count):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    
    return {
        "gpu_count": gpu_count,
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
    }

In [ ]:
def get_optimal_device_map(gpu_count):
    """GPU 수에 따른 최적의 device_map 반환"""
    if gpu_count <= 1:
        return "auto"
    
    # GPU가 2개 이상일 경우 자동으로 최적화된 device_map 생성
    return "auto"


In [ ]:
class CustomImageDataset(Dataset):
    """이미지 데이터셋 클래스 (간소화)"""
    def __init__(self, root_dir, full_image_dir, fold=None, split='train'):
        self.root_dir = root_dir
        self.full_image_dir = full_image_dir
        self.fold = fold
        self.split = split

        # 이미지 및 메타데이터 디렉토리 설정
        if fold is None:
            self.images_dir = os.path.join(root_dir, 'test', 'images')
            self.metadata_dir = os.path.join(root_dir, 'test', 'metadata')
        else:
            self.images_dir = os.path.join(root_dir, f'fold_{fold}', split, 'images')
            self.metadata_dir = os.path.join(root_dir, f'fold_{fold}', split, 'metadata')
        
        print(f"\n데이터셋 로드 중: {self.images_dir}")
        print(f"메타데이터: {self.metadata_dir}")
        
        # 메타데이터 로드
        self.metadata_cache = self._load_metadata()
        
        # 이미지 파일 로드
        self.images = self._load_images()
        
        print(f'\n데이터셋 로드 완료 ({split}): 총 {len(self.images)}개 샘플')
        self._print_label_stats()
    
    def _load_metadata(self):
        """메타데이터 로드"""
        metadata = {}
        json_files = [f for f in os.listdir(self.metadata_dir) if f.endswith('.json')]
        
        for json_file in json_files:
            try:
                with open(os.path.join(self.metadata_dir, json_file), 'r') as f:
                    data = json.load(f)
                filename = json_file.replace('.json', '')
                metadata[filename] = data
            except Exception as e:
                print(f"메타데이터 로드 오류 ({json_file}): {e}")
        
        return metadata
    
    def _load_images(self):
        """이미지 파일 로드"""
        images = []
        view_folders = ['Lateral', 'Left', 'Right']
        
        for view in view_folders:
            view_path = os.path.join(self.images_dir, view)
            if not os.path.exists(view_path):
                continue
                
            print(f"{view} 폴더 처리 중...")
            png_files = [f for f in os.listdir(view_path) if f.endswith('.png')]
            
            for img_name in png_files:
                filename = img_name.replace('.png', '')
                if filename in self.metadata_cache:
                    metadata = self.metadata_cache[filename]
                    label = metadata.get('label', None)
                    images.append({
                        'path': os.path.join(view_path, img_name),
                        'filename': img_name,
                        'label': label
                    })
        
        return images
    
    def _print_label_stats(self):
        """라벨 통계 출력"""
        if self.fold is not None and len(self.images) > 0:
            label_counts = {i: 0 for i in range(1, 5)}
            for img in self.images:
                if img['label'] is not None:
                    label_counts[img['label']] += 1
            
            for label, count in sorted(label_counts.items()):
                print(f' - Garden Type {label}: {count}')
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_info = self.images[idx]
        
        # 이미지 로드
        try:
            crop_image = Image.open(img_info['path']).convert('RGB')
        except Exception as e:
            print(f"이미지 로드 오류 ({img_info['path']}): {e}")
            crop_image = Image.new('RGB', (224, 224))
        
        # DICOM 파일 로드
        filename = img_info['filename']
        serial = filename.split('_')[0]
        full_img_path = os.path.join(self.full_image_dir, serial, f"{serial}t000.dcm")
        
        try:
            dcm = pydicom.dcmread(full_img_path)
            full_image_array = dcm.pixel_array.astype(np.float32)
            
            # 정규화
            min_val, max_val = np.min(full_image_array), np.max(full_image_array)
            if max_val != min_val:
                full_image_array = (full_image_array - min_val) / (max_val - min_val)
            else:
                full_image_array = np.zeros_like(full_image_array)
                
            full_image_array = (full_image_array * 255).astype(np.uint8)
            full_image = Image.fromarray(full_image_array).convert('RGB')
        except Exception as e:
            print(f'DICOM 로드 오류 ({full_img_path}): {e}')
            full_image = Image.new('RGB', (224, 224))
        
        # 라벨 처리
        label = img_info.get('label')
        label_tensor = torch.tensor(label-1, dtype=torch.long) if label is not None else None
        
        return {
            'crop_image': crop_image,
            'full_image': full_image,
            'filename': filename,
            'label': label_tensor,
            'path': img_info['path']
        }

In [ ]:

class CustomVQADataset(Dataset):
    """VQA 데이터셋 클래스 (간소화)"""
    def __init__(self, image_dataset, transform=None):
        self.image_dataset = image_dataset
        
        # 기본 변환 설정
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform
        
        # 표준 질문 템플릿
        self.question = """Analyze the type of femoral neck fracture (Garden classification) shown in this X-ray image and determine which Garden type it belongs to.\
            Consider the following criteria in your response:
            1: "This is a Garden Type I fracture. \
                    The key features are:\
                    - Incomplete fracture with valgus impaction,\
                    - Minimal or no cortical disruption,\
                    - Generally stable configuration.\
                    The fracture line may be subtle, often appearing as trabecular impaction rather than a clear break.",
                
                2: "This is a Garden Type II fracture.\
                    The characteristic features include:\
                    - Complete fracture without displacement,\
                    - Minimal disruption of trabecular pattern,\
                    - No significant angulation or rotation.\
                    Despite the complete fracture, the bone fragments remain properly aligned, making it a stable fracture.",
                
                3: "This is a Garden Type III fracture.\
                    The diagnostic features include:\
                    - Complete fracture with partial displacement,\
                    - Some disruption of trabecular alignment,\
                    - Cortical contact partially maintained but with angulation.\
                    There may be early signs of femoral head malalignment, increasing the risk of instability and avascular necrosis.",
                
                4: "This is a Garden Type IV fracture.\
                    The distinctive features include:\
                    - Complete fracture with full displacement,\
                    - No cortical contact between fragments,\
                    - Severe disruption of trabecular and anatomical alignment.\
                    The femoral head is completely separated from the shaft, significantly increasing the risk of avascular necrosis."""
    
    def __len__(self):
        return len(self.image_dataset)
    
    def __getitem__(self, idx):
        item = self.image_dataset[idx]
        
        # 이미지 변환
        crop_image = self.transform(item['crop_image'])
        full_image = self.transform(item['full_image'])
        label = item['label']
        
        # 답변 생성
        answer = f"This is a Garden Type {label+1} fracture."
        
        return {
            'crop_image': crop_image,
            'full_image': full_image,
            'question': self.question,
            'answer': answer,
            'label': label,
            'id': f"sample_{idx:06d}"
        }


In [ ]:
def load_model(config=None):
    """모델 로드 함수"""
    # 기본 설정이 없으면 생성
    if config is None:
        from dataclasses import dataclass
        
        @dataclass
        class ModelConfig:
            model_path: str = "microsoft/llava-med-v1.5-mistral-7b"
            max_length: int = 4096
            load_in_8bit: bool = True
            cache_dir: str = "/home/cbn-gpu12/FNF/VLM/LLavA/dataset/cache"
            use_safetensors: bool = True
        
        config = ModelConfig()
    
    
    # 시작 시간 기록 (이 부분이 빠져있었음)
    start_time = time.time()
    print(f"[{time.strftime('%H:%M:%S')}] 모델 로드 시작...")
    
    # GPU 개수 확인
    gpu_count = torch.cuda.device_count()
    print(f"사용 가능한 GPU: {gpu_count}개")
    
    # 메모리 설정 (이 부분이 빠져있었음)
    max_memory = {i: "12GB" for i in range(gpu_count)}
    max_memory["cpu"] = "24GB"
    
    # device_map 설정 (이 부분이 빠져있었음)
    device_map = "auto"
    
    # 양자화 설정 (이 부분이 빠져있었음)
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=config.load_in_8bit,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False
    )
    
    class ProgressCallback:
        def __init__(self):
            self.pbar = None
            self.start_time = time.time()
        
        def __call__(self, current, total):
            if self.pbar is None:
                self.pbar = tqdm(total=total, unit='B', unit_scale=True,
                                desc="모델 로드 중")
            
            # 진행바 업데이트
            self.pbar.update(current - self.pbar.n)
            
            # 진행률 및 속도 계산
            percent = (current / total) * 100
            elapsed = time.time() - self.start_time
            speed = current / (elapsed * 1024 * 1024) if elapsed > 0 else 0
            eta = (total - current) / (speed * 1024 * 1024) if speed > 0 else 0
            
            # 콘솔 출력
            print(f"\r[{time.strftime('%H:%M:%S')}] 로드 중: {percent:.1f}% | "
                  f"{current/(1024**3):.1f}/{total/(1024**3):.1f} GB | "
                  f"{speed:.1f} MB/s | ETA: {eta:.0f}초", end="")
            
            # 완료 시 출력
            if current >= total:
                total_time = time.time() - self.start_time
                print(f"\n[{time.strftime('%H:%M:%S')}] 모델 로드 완료! "
                      f"(소요 시간: {total_time:.1f}초, 평균 속도: {total/(total_time*1024*1024):.1f} MB/s)")
                if self.pbar:
                    self.pbar.close()

    try:
        print(f"[{time.strftime('%H:%M:%S')}] LLaVA-Med 모델 로드 중...")
        
        
        model = LlavaMistralForCausalLM.from_pretrained(
            config.model_path,
            device_map=device_map,
            max_memory=max_memory,
            quantization_config=quantization_config,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            use_safetensors=config.use_safetensors,
            cache_dir=config.cache_dir
        )
        
        # 그라디언트 체크포인팅 활성화
        if hasattr(model, 'gradient_checkpointing_enable'):
            model.gradient_checkpointing_enable()
        
        # 성능 측정
        elapsed = time.time() - start_time
        print(f"[{time.strftime('%H:%M:%S')}] 모델 로드 완료 (총 {elapsed:.2f}초)")
        
        # Vision Tower 로드
        print(f"[{time.strftime('%H:%M:%S')}] Vision Tower 로드 중...")
        vision_start = time.time()
        
        vision_tower = model.get_vision_tower()
        if not vision_tower.is_loaded:
            vision_tower.load_model()
        vision_tower.to(dtype=torch.float16)
        
        vision_elapsed = time.time() - vision_start
        print(f"[{time.strftime('%H:%M:%S')}] Vision Tower 로드 완료 ({vision_elapsed:.2f}초)")
        
        # 토크나이저 로드
        tokenizer = LlamaTokenizer.from_pretrained(
            config.model_path,
            use_fast=False,
            padding_side="right",
            model_max_length=config.max_length
        )
        
        return model, tokenizer, vision_tower
        
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] 모델 로드 오류: {e}")
        print(f"[{time.strftime('%H:%M:%S')}] 단순화된 설정으로 재시도...")
        
        # 더 단순한 설정으로 재시도
        model = LlavaMistralForCausalLM.from_pretrained(
            config.model_path,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        
        tokenizer = LlamaTokenizer.from_pretrained(
            config.model_path,
            use_fast=False,
            padding_side="right"
        )
        
        vision_tower = model.get_vision_tower()
        if not vision_tower.is_loaded:
            vision_tower.load_model()
        
        return model, tokenizer, vision_tower
    

In [ ]:
# 6. PEFT/LoRA 설정
def setup_peft_model(model):
    """간소화된 PEFT/LoRA 설정"""
    print(f"[{time.strftime('%H:%M:%S')}] LoRA 설정 적용 중...")
    start_time = time.time()
    
    # GPU 메모리 정리
    torch.cuda.empty_cache()
    gc.collect()
    
    # 기존 device_map 기억
    original_device_map = getattr(model, "hf_device_map", None)
    
    # LoRA 구성
    lora_config = LoraConfig(
        r=8,  # 더 효과적인 학습을 위해 더 높은 랭크 사용
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_alpha=16,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    # 모델 준비
    model = prepare_model_for_kbit_training(model)
    
    # PEFT 모델 생성
    peft_model = get_peft_model(model, lora_config)
    
    # 학습 설정
    peft_model.train()
    if hasattr(peft_model, 'enable_input_require_grads'):
        peft_model.enable_input_require_grads()
    
    # 완료 메시지
    elapsed = time.time() - start_time
    print(f"[{time.strftime('%H:%M:%S')}] LoRA 적용 완료 (소요 시간: {elapsed:.2f}초)")
    
    # 학습 가능한 파라미터 수 출력
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    print(f"학습 가능한 파라미터: {trainable_params:,} / 전체 파라미터: {total_params:,} ({trainable_params/total_params*100:.2f}%)")
    
    return peft_model

In [ ]:
def create_dataloaders(config):
    """데이터 로더 생성"""
    # 데이터셋 경로
    dataset_root = '/mnt/nas_backup/고효진/FNF/dataset'
    full_image_dir = '/mnt/nas_backup/고효진/FNF/FNF_origin/FNF/Final_data/datasets/raw_internal'
    
    # 배치 크기 및 워커 수
    batch_size = 8
    num_workers = 2
    
    train_loaders = {}
    val_loaders = {}
    
    print("데이터 로더 초기화 중...")
    start_time = time.time()
    
    # 각 폴드에 대한 데이터 로더 생성
    for fold in range(1, 6):
        # 훈련 데이터셋
        train_dataset = CustomImageDataset(
            root_dir=dataset_root,
            full_image_dir=full_image_dir,
            fold=fold,
            split='train'
        )
        train_vqa = CustomVQADataset(train_dataset)
        train_loader = DataLoader(
            train_vqa,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers
        )
        
        # 검증 데이터셋
        val_dataset = CustomImageDataset(
            root_dir=dataset_root,
            full_image_dir=full_image_dir,
            fold=fold,
            split='val'
        )
        val_vqa = CustomVQADataset(val_dataset)
        val_loader = DataLoader(
            val_vqa,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers
        )
        
        train_loaders[fold] = train_loader
        val_loaders[fold] = val_loader
    
    elapsed = time.time() - start_time
    print(f"데이터 로더 초기화 완료 (소요 시간: {elapsed:.2f}초)")
    
    return train_loaders, val_loaders

In [ ]:
class DataCollator:
    """데이터 콜레이터"""
    def __init__(self, tokenizer, split, conversation_template, pad_token_id, image_processor, model_config):
        self.tokenizer = tokenizer
        self.split = split
        self.conversation_template = conversation_template
        self.pad_token_id = pad_token_id
        self.image_processor = image_processor
        self.model_config = model_config
    
    def __call__(self, rows):
        if not isinstance(rows, list):
            rows = [rows]
        
        if self.split == "train":
            return self._collate_train(rows)
        elif self.split == "val":
            return self._collate_test(rows)

    def _collate_train(self, rows):
        train_input_ids_list = []
        train_labels_list = []
        train_images = []
        sample_ids = []

        for row in rows:
            try:
                # 이미지 처리는 CPU에서 수행
                with torch.no_grad():
                    crop_image = row['crop_image']
                    full_image = row['full_image']
                    combined_image = torch.cat([crop_image, full_image], dim=1)
                    
                    processed_image = combined_image.permute(1, 2, 0).cpu().numpy()
                    processed_image = Image.fromarray((processed_image * 255).astype(np.uint8))
                    processed_image = self.image_processor(
                        processed_image,
                        return_tensors="pt"
                    )["pixel_values"][0]
                    
                    train_images.append(processed_image)

                # 나머지 처리
                question = row['question']
                answer = row['answer']
                
                # 기존 텍스트 처리 로직 유지
                question = question.replace(DEFAULT_IMAGE_TOKEN, '').strip()
                question = DEFAULT_IMAGE_TOKEN + '\n' + question

                conv = self.conversation_template.copy()
                conv.append_message(conv.roles[0], question)
                conv.append_message(conv.roles[1], None)
                prefix = conv.get_prompt()

                conv = self.conversation_template.copy()
                conv.append_message(conv.roles[0], question)
                conv.append_message(conv.roles[1], answer)
                full = conv.get_prompt()

                prefix = tokenizer_image_token(prefix, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
                full = tokenizer_image_token(full, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")

                prefix_length = prefix.size(0)
                train_input_ids = full
                train_labels = full.clone()
                train_labels[:prefix_length] = -100

                train_input_ids_list.append(train_input_ids)
                train_labels_list.append(train_labels)
                sample_ids.append(row.get('id', -1))

            except Exception as e:
                print(f"Error processing row: {e}")
                continue

        if not train_input_ids_list:
            raise ValueError("No valid data found in the batch")

        # 패딩 처리
        train_input_ids = pad_sequence(train_input_ids_list, batch_first=True, padding_value=self.pad_token_id)
        train_labels = pad_sequence(train_labels_list, batch_first=True, padding_value=self.pad_token_id)
        train_attention_mask = (train_input_ids != self.pad_token_id).long()
        
        # 이미지 스택
        train_images = torch.stack(train_images)

        return {
            "input_ids": train_input_ids,
            "labels": train_labels,
            "attention_mask": train_attention_mask,
            "images": train_images,
            "metadata": {"sample_ids": sample_ids}
        }
    def _collate_test(self, rows):
        pass

In [ ]:
def main():
    """메인 함수"""
    # 기본 설정
    config = ModelConfig()
    
    # 모델 로드
    model, tokenizer, vision_tower = load_model(config)
    
    # LoRA 설정 적용
    peft_model = setup_peft_model(model)
    
    # 데이터 로더 생성
    train_loaders, val_loaders = create_dataloaders(config)
    
    # 학습 설정
    training_args = TrainingArguments(
        output_dir="trained_llava-med",
        report_to="wandb",
        run_name=f"fnf-classification-{datetime.now().strftime('%Y%m%d-%H%M')}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=16,
        learning_rate=2e-5,
        logging_steps=10,
        save_steps=100,
        evaluation_strategy="steps",
        eval_steps=100,
        save_total_limit=3,
        num_train_epochs=5,
        warmup_ratio=0.03,
        weight_decay=0.01,
        gradient_checkpointing=True,
        bf16=True,
        optim='paged_adamw_8bit',
        ddp_find_unused_parameters=False
    )
    
    # 학습 준비
    image_processor = vision_tower.image_processor
    conv = conv_templates["mistral_instruct"]
    
    # DataCollator 설정
    collate_fn = DataCollator(
        tokenizer=tokenizer,
        split="train",
        conversation_template=conv,
        pad_token_id=tokenizer.pad_token_id,
        image_processor=image_processor,
        model_config=model.config
    )
    
    print("모델 및 데이터 파이프라인 준비 완료!")
    print(f"GPU 메모리 사용량:")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / (1024**3)
        reserved = torch.cuda.memory_reserved(i) / (1024**3)
        print(f"GPU {i}: {allocated:.2f} GB 할당, {reserved:.2f} GB 예약")

# 스크립트 실행
if __name__ == "__main__":
    main()